# Component 2: cam multi-person YOLOv8 fine-tune (Google Colab)

Train a single-class person detector on `Dataset/yolo_cam_multiperson`, then download `yolov8n-cam-multiperson.pt`.

Set the Colab runtime to **GPU** before running (**Runtime → Change runtime type → T4 GPU**).

## Zip the dataset on your machine

From `member2-person-tracking`, after `python auto_label_cam_multiperson.py` and `python prepare_cam_multiperson.py`:

```powershell
tar -a -cf yolo_cam_multiperson.zip Dataset/yolo_cam_multiperson
```

Upload `yolo_cam_multiperson.zip` to `/content`. Unzipping it there produces `/content/Dataset/yolo_cam_multiperson/dataset.yaml`, which is the path used for training. Labels are class-0 boxes from YOLOv8n at confidence 0.25. `classes=[0]` drops images with no person box, and the unzip cell stops if every label file is empty.

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
import os
import zipfile
from pathlib import Path

# Drive upload: set ZIP_SRC to the Drive path and uncomment the mount.
# from google.colab import drive
# drive.mount("/content/drive")
# ZIP_SRC = "/content/drive/MyDrive/yolo_cam_multiperson.zip"

ZIP_SRC = "/content/yolo_cam_multiperson.zip"
EXTRACT_DIR = Path("/content")
if not os.path.exists(ZIP_SRC):
    raise FileNotFoundError(f"Upload the dataset zip to {ZIP_SRC}")
with zipfile.ZipFile(ZIP_SRC) as archive:
    archive.extractall(EXTRACT_DIR)

candidates = [
    EXTRACT_DIR / "Dataset" / "yolo_cam_multiperson",
    EXTRACT_DIR / "yolo_cam_multiperson",
]
dataset_root = next(path for path in candidates if (path / "images" / "train").is_dir())
yaml_path = dataset_root / "dataset.yaml"
yaml_path.write_text(
    "path: " + dataset_root.as_posix() + "\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: person\n",
    encoding="utf-8",
)
train_images = list((dataset_root / "images" / "train").glob("*.*"))
val_images = list((dataset_root / "images" / "val").glob("*.*"))
labelled = [
    path for path in (dataset_root / "labels").rglob("*.txt")
    if path.read_text(encoding="utf-8").strip()
]
print(f"train={len(train_images)} val={len(val_images)} labelled={len(labelled)}")
print(f"dataset yaml: {yaml_path}")
if not labelled:
    raise RuntimeError(
        "No person boxes found. Add class-0 YOLO labels and rebuild the zip before training. "
        "classes=[0] would otherwise remove every image."
    )

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Baseline person detector
results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    workers=4,
    classes=[0],  # Strictly restrict to class 0 (person)
    project="person_tracking_colab",
    name="yolov8n_cam_multiperson",
    save=True,
    exist_ok=True,
)

In [ ]:
import shutil
from pathlib import Path

from google.colab import files

best_pt = Path("person_tracking_colab/yolov8n_cam_multiperson/weights/best.pt")
if not best_pt.is_file():
    best_pt = Path(results.save_dir) / "weights" / "best.pt"
if not best_pt.is_file():
    raise FileNotFoundError(f"best.pt was not written: {best_pt}")
download_path = Path("/content/yolov8n-cam-multiperson.pt")
shutil.copy2(best_pt, download_path)
print(f"saved {best_pt} -> {download_path}")
files.download(str(download_path))
print("Place the downloaded file at member2-person-tracking/models/yolov8n-cam-multiperson.pt")